# BGE-large Baseline Retrieval
## Dense semantic search over 98,716 companies

**What this notebook does:**

Implements the BGE-large dense retrieval baseline — the strongest open-source
embedding model for retrieval tasks. Used as the primary dense comparison point.

**How dense retrieval works (plain English):**
Instead of matching keywords, we convert every company description into a list of
1,024 numbers (a *vector* or *embedding*) that captures its meaning. We do the
same for the query. Then we find companies whose vectors are closest to the query
vector — meaning they are semantically similar, even if they use different words.

**What is being embedded:**
All available company fields are combined into one rich text string per company —
name, country, state, city, industry (NACE), size, summary, and keywords.
This gives BGE full context when forming the embedding.

**Why BGE-large?**
- 335M parameters, 1024-dimensional embeddings
- Specifically trained for retrieval (hard negative mining)
- Requires `normalize_embeddings=True` and `IndexFlatIP` (NOT `IndexFlatL2`)
- Fastest dense method on GPU — faster than BM25 on CPU

**Folder structure:**
```
result/
└── 02_baseline_bge/
    ├── company_embeddings.npy    # Pre-computed BGE embeddings (98716 × 1024)
    ├── company_faiss.index       # FAISS index for fast search
    ├── bge_results.csv           # Top-1000 BGE results per query (101 queries)
    └── evaluation_bge.csv        # NDCG, Precision, Recall, F1 @ k∈{10,50,100,1000}
```

### Notebook structure
1. Environment setup
2. Imports
3. Load dataset & build corpus
4. GPU check
5. Encode all companies → embeddings
6. Build FAISS index
7. Run all 101 queries → results
8. Evaluation — NDCG, Precision, Recall, F1 @ k∈{10,50,100,1000}
9. Final summary

## 1 · Environment Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
API_KEY  = os.getenv('API_KEY')
BASE_URL = os.getenv('BASE_URL')

RESULT_DIR = Path('result/02_baseline_bge_summary')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Setup] Result folder: {RESULT_DIR}/ — ready')

[Setup] Result folder: result/02_baseline_bge/ — ready


## 2 · Imports

| Package | Role |
|---|---|
| `sentence_transformers` | Loads `BAAI/bge-large-en-v1.5` — produces 1024-d embeddings |
| `faiss` | Facebook AI Similarity Search — fast vector nearest-neighbour lookup |
| `torch` | PyTorch — used to verify GPU is available |
| `pandas / numpy` | Data loading and numerical operations |
| `time` | Measuring encoding and query latency |

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import torch
import json, time
import numpy as np
import pandas as pd
from pathlib import Path
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

RESULT_DIR = Path('result/02_baseline_bge_summary')
print('[Imports] All packages loaded successfully')

[Imports] All packages loaded successfully


## 3 · Load Dataset & Build Corpus

Same source as the BM25 notebook — `dataset/production_results.xlsx`.
We build the same rich text representation per company so that the BM25 and BGE
baselines index the same content and results are directly comparable.

**Why combine all fields into one string for embedding?**
BGE encodes a single text string. By including country, industry, size, and keywords
alongside the summary, the resulting embedding captures all of that context.
A query for *"software companies in Germany"* will then match companies whose
embedding reflects both "software" AND "Germany" — not just one or the other.

In [ ]:
print('[Load] Loading production results...')
results_df = pd.read_excel('dataset/production_results.xlsx')
print(f'[Load] Total rows        : {len(results_df):,}')
print(f'[Load] Columns available : {list(results_df.columns)}')

all_companies = results_df.drop_duplicates(subset='domain').reset_index(drop=True)
print(f'[Load] Unique companies  : {len(all_companies):,}')

with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)
print(f'[Load] Queries           : {len(data)}')

# Build rich text per company
print('[Load] Building rich text for each company...')

def build_rich_text(row):
    """Summary only — Set A baseline experiments."""
    return str(row.get('summary', '')) if pd.notna(row.get('summary')) else ''

rich_texts = [build_rich_text(row) for _, row in all_companies.iterrows()]

print(f'[Load] Sample rich text (first company):')
print(f'  {rich_texts[0][:300]}...')

[Load] Loading production results...
[Load] Total rows        : 101,000
[Load] Columns available : ['rank', 'similarity_score', 'domain', 'name', 'organization_type', 'organization_size', 'country', 'state', 'district', 'municipality', 'summary', 'summary_keywords', 'nace_code', 'query_id', 'query']
[Load] Unique companies  : 98,716
[Load] Queries           : 101
[Load] Building rich text for each company...
[Load] Sample rich text (first company):
  Company: Software Genesis, Inc. | Country: United States | State: Illinois | Type: Company | Size: Micro (0-9) | Industry: NACE K: Telecommunication, computer programming, consulting, computing infrastructure and other information service activities | Software Genesis is a software development compa...


## 4 · GPU Check

BGE-large encoding **requires GPU** to be practical.
- On CPU: ~30-40 minutes for 99k companies
- On A100 GPU: ~7 minutes

The query encoding at search time also runs on GPU — this is why BGE achieves
44ms query latency, faster than BM25's 104ms on CPU.

In [4]:
print(f'[GPU] CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[GPU] Device          : {torch.cuda.get_device_name(0)}')
    print(f'[GPU] VRAM            : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('[GPU] WARNING: No GPU — encoding will take 30-40 minutes on CPU')
    DEVICE = 'cpu'
print(f'[GPU] Using device    : {DEVICE}')

[GPU] CUDA available  : False
[GPU] WARNING: No GPU — encoding will take 30-40 minutes on CPU
[GPU] Using device    : cpu


## 5 · Encode All Companies with BGE-large

**Model:** `BAAI/bge-large-en-v1.5`

Each company's rich text is converted into a 1,024-dimensional vector.
Vectors are L2-normalised (`normalize_embeddings=True`) — required for BGE
because it uses cosine similarity. Normalised vectors allow us to use
`IndexFlatIP` (inner product) which equals cosine similarity for unit vectors.

**Embeddings are saved immediately** to avoid re-running this step
if the kernel restarts — encoding takes several minutes.

In [5]:
print('[Encode] Loading BGE-large model...')
t0    = time.time()
model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=DEVICE)
print(f'[Encode] Model loaded in  : {time.time()-t0:.1f}s  on {model.device}')

print('[Encode] Encoding all companies...')
print('[Encode] This takes ~7 minutes on A100 GPU')
t0 = time.time()

embeddings = model.encode(
    rich_texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # REQUIRED for BGE — uses cosine similarity
)

ENCODE_TIME = time.time() - t0
print(f'[Encode] Done in          : {ENCODE_TIME/60:.1f} minutes')
print(f'[Encode] Embeddings shape : {embeddings.shape}')  # (98716, 1024)

np.save(RESULT_DIR / 'company_embeddings.npy', embeddings)
print(f'[Encode] Saved to         : result/02_baseline_bge/company_embeddings.npy')

[Encode] Loading BGE-large model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Encode] Model loaded in  : 2.9s  on cpu
[Encode] Encoding all companies...
[Encode] This takes ~7 minutes on A100 GPU


Batches:   0%|          | 0/386 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6 · Build FAISS Index

**FAISS** (Facebook AI Similarity Search) is the library used to efficiently
find the nearest neighbour vectors for a query.

**Index type: `IndexFlatIP`** (Flat Inner Product)
- Exact search — guaranteed to find the true nearest neighbours
- Uses inner product distance which equals cosine similarity for normalised vectors
- Fast enough for 99k companies (~28ms search time)

> **Why NOT `IndexFlatL2`?**
> `IndexFlatL2` uses Euclidean distance. For normalised BGE embeddings,
> this gives wrong rankings. Always use `IndexFlatIP` with BGE.

In [ ]:
print('[FAISS] Loading embeddings...')
embeddings = np.load(RESULT_DIR / 'company_embeddings.npy').astype('float32')
print(f'[FAISS] Embeddings shape  : {embeddings.shape}')

print('[FAISS] Building index...')
t0        = time.time()
dimension = embeddings.shape[1]  # 1024
index     = faiss.IndexFlatIP(dimension)
index.add(embeddings)
INDEX_BUILD_TIME = time.time() - t0
print(f'[FAISS] Index built in    : {INDEX_BUILD_TIME:.2f}s')
print(f'[FAISS] Vectors in index  : {index.ntotal:,}')

faiss.write_index(index, str(RESULT_DIR / 'company_faiss.index'))
print(f'[FAISS] Index saved to    : result/02_baseline_bge/company_faiss.index')

## 7 · Run BGE Across All 101 Queries

For each query:
1. Encode the query string with BGE (normalised)
2. Search the FAISS index for top-1,000 nearest companies
3. Store results with rank and cosine similarity score

**Query latency** is measured per query — encode time + search time combined.

**Output:** `result/02_baseline_bge/bge_results.csv`

In [ ]:
print(f'[Run] Starting BGE retrieval for {len(data)} queries...')
print(f'[Run] Retrieving top-1000 per query')
print('-' * 55)

all_bge_results = []
query_times     = []
total_start     = time.time()

for i, item in enumerate(data):
    query_id = item['query_id']
    query    = item['query']

    # Encode query + search — time both together 
    t0            = time.perf_counter()
    query_emb     = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    scores, idxs  = index.search(query_emb, 1000)
    query_ms      = (time.perf_counter() - t0) * 1000
    query_times.append(query_ms)

    for rank, (idx, score) in enumerate(zip(idxs[0], scores[0])):
        company = all_companies.iloc[idx]
        all_bge_results.append({
            'query_id': query_id,
            'query':    query,
            'rank':     rank + 1,
            'score':    float(score),
            'domain':   company['domain'],
            'name':     company.get('name', ''),
            'country':  company.get('country', ''),
            'summary':  company.get('summary', ''),
        })

    if (i + 1) % 20 == 0 or (i + 1) == len(data):
        elapsed   = time.time() - total_start
        remaining = (len(data) - i - 1) * elapsed / (i + 1)
        print(f'[Run] {i+1:3d}/{len(data)}  |  '
              f'avg {sum(query_times)/len(query_times):.1f}ms/query  |  '
              f'~{remaining:.0f}s remaining')

bge_df = pd.DataFrame(all_bge_results)
bge_df.to_csv(RESULT_DIR / 'bge_results.csv', index=False)

AVG_LATENCY_MS = sum(query_times) / len(query_times)
print('-' * 55)
print(f'[Run] Done!')
print(f'[Run] Total results       : {len(bge_df):,}')
print(f'[Run] Avg query latency   : {AVG_LATENCY_MS:.1f}ms')
print(f'[Run] Saved to            : result/02_baseline_bge/bge_results.csv')

## 8 · Evaluation — NDCG, Precision, Recall, F1 @ k

Same evaluation protocol as the BM25 notebook so results are directly comparable.

### Pseudo-relevance labels
A company is **relevant** for a query if it appears in the **production top-100**.

### Metrics

| Metric | Formula | What it measures |
|---|---|---|
| **Precision@k** | \|Retrieved ∩ Relevant\| / k | Of the top-k results, what fraction are relevant? |
| **Recall@k** | \|Retrieved ∩ Relevant\| / \|Relevant\| | Of all relevant companies, what fraction did we find? |
| **F1@k** | 2 × P × R / (P + R) | Harmonic mean — balances Precision and Recall |
| **NDCG@k** | DCG@k / IDCG@k | Ranking quality — rewards relevant results ranked higher |

In [ ]:
print('[Eval] Loading production labels...')
production_df = pd.read_excel('dataset/production_results.xlsx')

K_VALUES = [10, 50, 100, 500,  1000]

def get_relevant(query_id, top_k=100):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

def f1_at_k(retrieved, relevant, k):
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

def dcg_at_k(retrieved, relevant, k):
    return sum(
        1 / np.log2(i + 2)
        for i, d in enumerate(retrieved[:k]) if d in relevant
    )

def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0

# Evaluate all queries
print('[Eval] Computing metrics for all queries...')
eval_rows = []

for i, item in enumerate(data):
    qid       = item['query_id']
    query     = item['query']
    relevant  = get_relevant(qid)
    retrieved = (
        bge_df[bge_df['query_id'] == qid]
        .sort_values('rank')['domain'].tolist()
    )
    for k in K_VALUES:
        eval_rows.append({
            'query_id':  qid,
            'query':     query,
            'k':         k,
            'precision': precision_at_k(retrieved, relevant, k),
            'recall':    recall_at_k(retrieved, relevant, k),
            'f1':        f1_at_k(retrieved, relevant, k),
            'ndcg':      ndcg_at_k(retrieved, relevant, k),
        })

    if (i + 1) % 25 == 0:
        print(f'[Eval] {i+1}/101 queries evaluated...')

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(RESULT_DIR / 'evaluation_bge.csv', index=False)
print(f'[Eval] Saved to result/02_baseline_bge/evaluation_bge.csv')

## 9 · Final Summary

Full results table averaged across all 101 queries.
Compare against the BM25 baseline from `01_baseline_bm25`.

In [ ]:
print('[Summary] ============================================================')
print('[Summary] BGE-large BASELINE RESULTS')
print(f'\n[Summary] Encoding time     : {ENCODE_TIME/60:.1f} minutes')
print(f'[Summary] Index build time  : {INDEX_BUILD_TIME:.2f}s')
print(f'[Summary] Avg query latency : {AVG_LATENCY_MS:.1f}ms')
print(f'[Summary] Companies encoded : {len(all_companies):,}')
print(f'[Summary] Embedding dims    : 1024')
print(f'[Summary] Queries evaluated : {len(data)}')
print()
print(f'  {"k":<6} {"NDCG":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-' * 46)
for k in K_VALUES:
    sub = eval_df[eval_df['k'] == k]
    print(f'  {k:<6} '
          f'{sub["ndcg"].mean():>8.3f} '
          f'{sub["precision"].mean():>10.3f} '
          f'{sub["recall"].mean():>8.3f} '
          f'{sub["f1"].mean():>8.3f}')